# 16 · `gl_engine/rating/kernel.py`

## What this file is for

**Submission in, rating out.** This is the front door, and everything in notebooks 01–15 exists so that this file can be short.

It resolves the packages, maps the submission onto the tree, runs the interpreter, collects the premium, and hands back a `Rating` — the number, the evidence, and the refusals.

**Depends on:** everything above it.

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.rating import kernel

for name, obj in vars(kernel).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != kernel.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Three lines, from a file on disk to a premium.

In [ ]:
from gl_engine.rating import Kernel

rating = Kernel().rate("../Engine_Payloads/GA/submission.json")

print("premium  :", rating.premium)
print("packages :", rating.packages)
print("complete :", rating.complete)
print("trace     :", len(rating.trace), "entries")

That is the whole public API of the engine. A notebook, a script and the interface all call exactly this.

## The interesting case

### The premium is not the interesting part

A `Rating` carries the evidence with it, which is what makes a disputed premium answerable.

In [ ]:
print("premium      :", rating.premium)
print("by_coverage  :", len(rating.by_coverage), "coverage groups")
for name, value in list(rating.by_coverage.items())[:4]:
    print(f"    {name:<58} {value}")

print("\nISO's own messages:", len(rating.messages))
for m in rating.messages[:2]:
    print("   ", m)

print("\nreferrals    :", len(rating.referrals))
print("stopped      :", rating.stopped)

### The trace is every step, in order

Two thousand entries for one submission. This is what the interface renders and what made the engine debuggable.

In [ ]:
print(f"{len(rating.trace)} trace entries. The first few:\n")
for entry in rating.trace[:10]:
    print("  ", entry)

### Two modes, one code path

`strict-erc` executes ISO's content and nothing else — it is the mode that stays permanently comparable against ISO's own service. `underwriting` adds the referral policy on top.

In [ ]:
from gl_engine.rating.kernel import MODES, STRICT, UNDERWRITING

print("modes:", MODES)

strict = Kernel(mode=STRICT).rate("../Engine_Payloads/GA/submission.json")
uw     = Kernel(mode=UNDERWRITING).rate("../Engine_Payloads/GA/submission.json")

print(f"\nstrict-erc    premium {strict.premium}  referrals {len(strict.referrals)}")
print(f"underwriting  premium {uw.premium}  referrals {len(uw.referrals)}")
print("\nSame number. The modes differ in what they REFUSE, not in what they compute.")

Keeping `strict-erc` permanently runnable is what allows the ISO baseline to go on being verified forever — including after carrier deviations exist on top of it.

## What it refuses

A submission it cannot rate does not produce a partial number.

In [ ]:
try:
    Kernel().rate({"nothing": "useful"})
    print("no error")
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:150]}")

## Try it yourself

1. Rate two different states and compare `by_coverage`. Which parts move, and which don't?
2. Set `rounding="ROUND_DOWN"` and rate. Does the premium change? (Notebook 12 has the context.)
3. Search the trace for `lookup-miss`. Each one is a state asking for a value it doesn't file.

In [ ]:
# your turn